# Food-Sure: Fair Comparison — ResNet50 vs CatBoost Pipeline

**Why this notebook exists.** The deployed ResNet50 reports 62.5–82.1% test accuracy on a **7-class** task
(pure + 2 adulterants × 5/10/15%). The newer CatBoost/ensemble pipeline (`Food_Sure_Model_Pipelines.ipynb`)
reports ~85% on a **3-class** task (pure / adulterant A / adulterant B), on a different split. Those numbers
cannot be compared directly. This notebook puts both models on equal footing.

**What makes it fair**

| | ResNet50 | CatBoost pipeline |
|---|---|---|
| Test images | `dataset_split/<spice>/test` (the original held-out set) | **same images** |
| Training images | train + val (val used for checkpoint selection) | **same train + val images** |
| 7-class task | native output | trained on 7 classes |
| 3-class task | 7-class probabilities summed per adulterant | trained on 3 classes (as in the original notebook) |
| Prediction unit | one photo | one photo (4 crops averaged, as in the original notebook) |
| Metrics | accuracy, macro-F1, pure recall, bootstrap 95% CI | same, plus paired McNemar test |

**Read-only with respect to existing work.** The ResNet50 models are loaded from `web_models/` (the exact
deployed models) and nothing in the Drive folder is modified. Results are written to a new
`Food Sure/fair_comparison/` folder. The CatBoost pipeline code below is copied unchanged from
`Food_Sure_Model_Pipelines.ipynb` (`extract_80plus_features`, routing: Pipeline 2 for red chilli, Pipeline 1 otherwise).

## 1 — Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q catboost lightgbm

In [ ]:
import os, re, json, warnings
import numpy as np, pandas as pd, cv2
import tensorflow as tf
from tqdm import tqdm
from scipy.stats import skew, kurtosis, binomtest
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.preprocessing import RobustScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score, confusion_matrix
import catboost as cb
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

BASE_DIR       = '/content/drive/MyDrive/Food Sure'
SPLIT_DIR      = os.path.join(BASE_DIR, 'dataset_split')
WEB_MODELS_DIR = os.path.join(BASE_DIR, 'web_models')
OUT_DIR        = os.path.join(BASE_DIR, 'fair_comparison')   # new folder, only outputs go here
os.makedirs(OUT_DIR, exist_ok=True)

SPICES = ['black_pepper_powder', 'red_chilli_powder', 'turmeric']
SEED = 42
CROP_SIZE, CROPS_PER_IMAGE = 256, 4
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp')

for s in SPICES:
    for part in ['train', 'val', 'test']:
        assert os.path.isdir(os.path.join(SPLIT_DIR, s, part)), f'missing {s}/{part}'
    assert os.path.exists(os.path.join(WEB_MODELS_DIR, s, 'model.keras')), f'missing ResNet50 for {s}'
print('✅ Paths OK | TF', tf.__version__)

## 2 — Data and label helpers

In [ ]:
def to_type(label):
    # '10_pct_rice_flour' -> 'rice_flour', 'pure' -> 'pure'  (the 3-class view)
    label = str(label).lower()
    return 'pure' if label == 'pure' else re.sub(r'^\d+_pct_', '', label)

def list_split(spice, part):
    # [(path, 7-class label)] for one split, sorted so runs are reproducible
    root = os.path.join(SPLIT_DIR, spice, part)
    items = []
    for cls in sorted(os.listdir(root)):
        d = os.path.join(root, cls)
        if not os.path.isdir(d):
            continue
        for f in sorted(os.listdir(d)):
            if f.lower().endswith(VALID_EXTS):
                items.append((os.path.join(d, f), cls.lower()))
    return items

for s in SPICES:
    tv, te = list_split(s, 'train') + list_split(s, 'val'), list_split(s, 'test')
    print(f'{s:<22} train+val={len(tv):>3}  test={len(te):>3}  classes={sorted({l for _, l in te})}')

## 3 — CatBoost pipeline (copied unchanged from `Food_Sure_Model_Pipelines.ipynb`)
Feature extractor `extract_80plus_features`, 4 crops per photo, RobustScaler → SelectKBest(k=65) →
Pipeline 1 (tuned CatBoost) for pepper & turmeric, Pipeline 2 (gray-world + CatBoost/LightGBM/ExtraTrees
soft vote) for red chilli. Crop probabilities are averaged per photo.

In [ ]:
def extract_80plus_features(img_bgr, use_gray_world=False):
    if use_gray_world:
        b, g, r = cv2.split(img_bgr.astype(np.float32))
        gray_avg = (np.mean(b) + np.mean(g) + np.mean(r)) / 3.0
        b = np.clip(b * (gray_avg / (np.mean(b) + 1e-5)), 0, 255)
        g = np.clip(g * (gray_avg / (np.mean(g) + 1e-5)), 0, 255)
        r = np.clip(r * (gray_avg / (np.mean(r) + 1e-5)), 0, 255)
        img_bgr = cv2.merge([b, g, r]).astype(np.uint8)

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    img_lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    img_ycrcb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2YCrCb)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    feats = []
    # A. Colour moments across 12 channels
    for space in [img_rgb, img_hsv, img_lab, img_ycrcb]:
        for ch in range(3):
            plane = space[:, :, ch].astype(np.float32).ravel()
            mean_v, std_v = np.mean(plane), np.std(plane)
            sk_v = float(skew(plane)) if std_v > 1e-3 else 0.0
            kt_v = float(kurtosis(plane)) if std_v > 1e-3 else 0.0
            feats.extend([mean_v, std_v, sk_v, kt_v])

    # B. Normalised histograms (H, S, L*, a*, b*)
    h_hist, _ = np.histogram(img_hsv[:, :, 0], bins=16, range=(0, 180), density=True)
    s_hist, _ = np.histogram(img_hsv[:, :, 1], bins=16, range=(0, 256), density=True)
    l_hist, _ = np.histogram(img_lab[:, :, 0], bins=16, range=(0, 256), density=True)
    a_hist, _ = np.histogram(img_lab[:, :, 1], bins=16, range=(0, 256), density=True)
    b_hist, _ = np.histogram(img_lab[:, :, 2], bins=16, range=(0, 256), density=True)
    feats.extend(np.hstack([h_hist, s_hist, l_hist, a_hist, b_hist]))

    # C. Domain physical indices
    lab_l = img_lab[:, :, 0].astype(np.float32)
    lab_a = img_lab[:, :, 1].astype(np.float32)
    lab_b = img_lab[:, :, 2].astype(np.float32)
    hsv_s = img_hsv[:, :, 1].astype(np.float32)
    cai = np.mean(lab_b) / (np.mean(lab_l) + 1e-5)
    rpi = np.mean(lab_a) / (np.mean(hsv_s) + 1e-5)
    fleck_ratio = np.sum(img_gray > 140) / (img_gray.size + 1e-5)
    lap_var = float(np.var(cv2.Laplacian(img_gray, cv2.CV_64F)))
    feats.extend([cai, rpi, fleck_ratio, lap_var])

    # D. Isotropic GLCM
    glcm = graycomatrix(img_gray, distances=[1, 2], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                        levels=256, symmetric=True, normed=True)
    glcm_iso = np.mean(glcm, axis=3, keepdims=True)
    for prop in ['contrast', 'dissimilarity', 'homogeneity', 'energy', 'correlation']:
        feats.extend(graycoprops(glcm_iso, prop).ravel())

    # E. LBP
    lbp = local_binary_pattern(img_gray, P=8, R=1, method='uniform')
    lbp_hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)
    feats.extend(lbp_hist)
    return np.array(feats, dtype=np.float32)


def image_crops(img):
    h, w = img.shape[:2]
    if h >= CROP_SIZE and w >= CROP_SIZE:
        coords = [((h - CROP_SIZE) // 2, (w - CROP_SIZE) // 2), (0, 0),
                  (h - CROP_SIZE, w - CROP_SIZE), (0, w - CROP_SIZE)]
        return [img[t:t + CROP_SIZE, l:l + CROP_SIZE] for t, l in coords[:CROPS_PER_IMAGE]]
    return [cv2.resize(img, (CROP_SIZE, CROP_SIZE))]


def featurize(spice, part_name, items, use_gray_world):
    # Features per crop; groups[i] = index of the source photo in `items`. Cached to OUT_DIR.
    cache = os.path.join(OUT_DIR, f'feats_{spice}_{part_name}_gw{int(use_gray_world)}.npz')
    if os.path.exists(cache):
        d = np.load(cache, allow_pickle=True)
        return d['X'], d['y'], d['groups']
    X, y, groups = [], [], []
    for gid, (p, label) in enumerate(tqdm(items, desc=f'{spice}/{part_name}')):
        img = cv2.imread(p)
        if img is None:
            print('⚠️ unreadable, skipped:', p)
            continue
        for patch in image_crops(img):
            X.append(extract_80plus_features(patch, use_gray_world=use_gray_world))
            y.append(label)
            groups.append(gid)
    X, y, groups = np.array(X), np.array(y), np.array(groups)
    np.savez(cache, X=X, y=y, groups=groups)
    return X, y, groups


def make_clf(spice):
    if spice == 'red_chilli_powder':  # Pipeline 2
        m1 = cb.CatBoostClassifier(iterations=550, learning_rate=0.035, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)
        m2 = lgb.LGBMClassifier(n_estimators=350, learning_rate=0.035, max_depth=5, class_weight='balanced', random_state=42, verbose=-1)
        m3 = ExtraTreesClassifier(n_estimators=300, max_depth=12, class_weight='balanced', random_state=42)
        return VotingClassifier(estimators=[('cb', m1), ('lgb', m2), ('et', m3)], voting='soft')
    # Pipeline 1
    return cb.CatBoostClassifier(iterations=600, learning_rate=0.04, depth=5, auto_class_weights='Balanced', verbose=0, random_seed=42)


def catboost_fit_predict(spice, X_tr, y_tr, X_te, groups_te):
    # Returns (photo ids, class names, per-photo probabilities averaged over its crops)
    scaler = RobustScaler().fit(X_tr)
    X_tr_s, X_te_s = scaler.transform(X_tr), scaler.transform(X_te)
    selector = SelectKBest(score_func=f_classif, k=min(65, X_tr.shape[1])).fit(X_tr_s, y_tr)
    clf = make_clf(spice)
    clf.fit(selector.transform(X_tr_s), y_tr)
    probs = clf.predict_proba(selector.transform(X_te_s))
    ids = np.unique(groups_te)
    photo_probs = np.stack([probs[groups_te == g].mean(axis=0) for g in ids])
    return ids, np.array(clf.classes_).astype(str), photo_probs

## 4 — ResNet50 (the deployed models from `web_models/`)
Images are loaded exactly as during training/evaluation (`load_img`, 224×224, nearest interpolation, raw 0–255
pixels; the model applies `resnet50.preprocess_input` internally). For the 3-class view, the 7 class
probabilities are summed per adulterant.

In [ ]:
def resnet_predict(spice, items):
    with open(os.path.join(WEB_MODELS_DIR, spice, 'label_map.json')) as f:
        label_map = json.load(f)
    classes = np.array([label_map[str(i)].lower() for i in range(len(label_map))])
    model = tf.keras.models.load_model(os.path.join(WEB_MODELS_DIR, spice, 'model.keras'))
    batch = np.stack([tf.keras.utils.img_to_array(tf.keras.utils.load_img(p, target_size=(224, 224)))
                      for p, _ in items])
    probs = model.predict(batch, batch_size=16, verbose=0)
    del model
    tf.keras.backend.clear_session()
    return classes, probs


def collapse_to_type(classes, probs):
    # Sum 7-class probabilities into pure / adulterant A / adulterant B
    types = np.array([to_type(c) for c in classes])
    tclasses = np.unique(types)
    return tclasses, np.stack([probs[:, types == t].sum(axis=1) for t in tclasses], axis=1)

## 5 — Metrics and paired statistics

In [ ]:
def score(y_true, y_pred):
    return {
        'acc':         accuracy_score(y_true, y_pred) * 100,
        'macro_f1':    f1_score(y_true, y_pred, average='macro', zero_division=0) * 100,
        'pure_recall': recall_score(y_true, y_pred, labels=['pure'], average='macro', zero_division=0) * 100,
    }

def boot_ci(correct, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(correct), (n, len(correct)))
    return np.percentile(correct[idx].mean(axis=1) * 100, [2.5, 97.5])

def boot_diff_ci(correct_a, correct_b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(correct_a), (n, len(correct_a)))
    return np.percentile((correct_a[idx].mean(axis=1) - correct_b[idx].mean(axis=1)) * 100, [2.5, 97.5])

def mcnemar(correct_a, correct_b):
    # Exact McNemar: only photos where the two models disagree count
    b = int(np.sum(correct_a & ~correct_b))   # A right, B wrong
    c = int(np.sum(~correct_a & correct_b))   # A wrong, B right
    p = 1.0 if b + c == 0 else binomtest(b, b + c, 0.5).pvalue
    return p, b, c

## 6 — Head-to-head on the same held-out test photos
Expect the ResNet50 7-class accuracy here to match the original numbers (82.14 / 62.5 / 78.95%) — that is the
sanity check that we are evaluating the same models on the same test set. Feature extraction is the slow part
(a few minutes per spice on CPU); features are cached in `fair_comparison/` so re-runs are fast.

In [ ]:
ORIGINAL_RESNET = {'black_pepper_powder': 82.14, 'red_chilli_powder': 62.5, 'turmeric': 78.95}
rows, paired, store = [], [], {}

for spice in SPICES:
    gw = (spice == 'red_chilli_powder')
    train_items = list_split(spice, 'train') + list_split(spice, 'val')
    test_items  = list_split(spice, 'test')

    X_tr, y_tr7, _    = featurize(spice, 'trainval', train_items, gw)
    X_te, _,     g_te = featurize(spice, 'test', test_items, gw)
    y_tr3 = np.array([to_type(l) for l in y_tr7])

    # CatBoost pipeline, trained natively on each task
    ids, c_cls7, c_p7 = catboost_fit_predict(spice, X_tr, y_tr7, X_te, g_te)
    _,   c_cls3, c_p3 = catboost_fit_predict(spice, X_tr, y_tr3, X_te, g_te)

    # ResNet50 on the same photos (restricted to photos the feature pipeline could read)
    r_cls7, r_p7 = resnet_predict(spice, test_items)
    r_p7 = r_p7[ids]
    r_cls3, r_p3 = collapse_to_type(r_cls7, r_p7)

    y_true7 = np.array([test_items[i][1] for i in ids])
    y_true3 = np.array([to_type(l) for l in y_true7])

    preds = {
        ('7-class', 'ResNet50'):          r_cls7[r_p7.argmax(1)],
        ('7-class', 'CatBoost pipeline'): c_cls7[c_p7.argmax(1)],
        ('3-class', 'ResNet50'):          r_cls3[r_p3.argmax(1)],
        ('3-class', 'CatBoost pipeline'): c_cls3[c_p3.argmax(1)],
    }
    store[spice] = {'y_true7': y_true7, 'y_true3': y_true3, 'preds': preds}

    r7 = accuracy_score(y_true7, preds[('7-class', 'ResNet50')]) * 100
    flag = '✅' if abs(r7 - ORIGINAL_RESNET[spice]) < 2.0 else '⚠️ differs from original — check before trusting'
    print(f'{spice}: ResNet50 7-class here = {r7:.2f}% vs original {ORIGINAL_RESNET[spice]}%  {flag}')

    for task in ['7-class', '3-class']:
        y_true = y_true7 if task == '7-class' else y_true3
        correct = {}
        for model in ['ResNet50', 'CatBoost pipeline']:
            yp = preds[(task, model)]
            correct[model] = (yp == y_true)
            lo, hi = boot_ci(correct[model])
            rows.append({'spice': spice, 'task': task, 'model': model, 'n_test_photos': len(y_true),
                         **score(y_true, yp), 'acc_ci_low': lo, 'acc_ci_high': hi})
        p, b, c = mcnemar(correct['ResNet50'], correct['CatBoost pipeline'])
        dlo, dhi = boot_diff_ci(correct['CatBoost pipeline'], correct['ResNet50'])
        paired.append({'spice': spice, 'task': task,
                       'CatBoost_minus_ResNet_pp': (correct['CatBoost pipeline'].mean() - correct['ResNet50'].mean()) * 100,
                       'diff_ci_low': dlo, 'diff_ci_high': dhi,
                       'only_ResNet_right': b, 'only_CatBoost_right': c, 'mcnemar_p': p})

results = pd.DataFrame(rows).round(2)
paired_df = pd.DataFrame(paired).round(3)
results.to_csv(os.path.join(OUT_DIR, 'head_to_head.csv'), index=False)
paired_df.to_csv(os.path.join(OUT_DIR, 'paired_tests.csv'), index=False)
display(results)
display(paired_df)

### Summary table and chart

In [ ]:
pivot = results.pivot_table(index='spice', columns=['task', 'model'], values='acc')
display(pivot.round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, task in zip(axes, ['7-class', '3-class']):
    sub = results[results.task == task]
    x = np.arange(len(SPICES)); w = 0.38
    for k, (model, color) in enumerate([('ResNet50', '#9467bd'), ('CatBoost pipeline', '#2ca02c')]):
        m = sub[sub.model == model].set_index('spice').loc[SPICES]
        err = [m.acc - m.acc_ci_low, m.acc_ci_high - m.acc]
        bars = ax.bar(x + (k - 0.5) * w, m.acc, w, yerr=err, capsize=4, label=model, color=color)
        for bar, v in zip(bars, m.acc):
            ax.annotate(f'{v:.1f}%', (bar.get_x() + bar.get_width() / 2, v), xytext=(0, 3),
                        textcoords='offset points', ha='center', fontsize=9, fontweight='bold')
    chance = 100 / 7 if task == '7-class' else 100 / 3
    ax.axhline(chance, ls='--', c='gray', lw=1, label=f'chance ({chance:.1f}%)')
    ax.set_xticks(x); ax.set_xticklabels(['Black Pepper', 'Red Chilli', 'Turmeric'])
    ax.set_title(f'{task}: same test photos, 95% bootstrap CI'); ax.set_ylim(0, 110)
    ax.grid(axis='y', alpha=0.3)
axes[0].set_ylabel('Photo-level test accuracy (%)'); axes[1].legend(loc='lower right')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'fair_comparison_chart.png'), dpi=200)
plt.show()

### Confusion matrices (3-class view)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for j, spice in enumerate(SPICES):
    s = store[spice]
    labels = sorted(np.unique(s['y_true3']))
    for i, model in enumerate(['ResNet50', 'CatBoost pipeline']):
        yp = s['preds'][('3-class', model)]
        cm = confusion_matrix(s['y_true3'], yp, labels=labels)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Purples' if i == 0 else 'Greens', cbar=False,
                    xticklabels=labels, yticklabels=labels, ax=axes[i, j])
        axes[i, j].set_title(f'{model} — {spice.replace("_", " ")} ({accuracy_score(s["y_true3"], yp)*100:.1f}%)')
        axes[i, j].set_xlabel('Predicted'); axes[i, j].set_ylabel('Actual')
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'confusion_3class.png'), dpi=200)
plt.show()

## 7 — How stable is the CatBoost "~85%"? (5-fold CV over all photos)
The original notebook reports a single fold (`next(sgkf.split(...))`). Here the same pipeline is run on 5
stratified folds over every photo (crops of one photo always stay together), for both tasks, so we get a
mean ± std instead of one number. Uses the cached features — no re-extraction.

In [ ]:
cv_rows = []
for spice in SPICES:
    gw = (spice == 'red_chilli_powder')
    tv_items = list_split(spice, 'train') + list_split(spice, 'val')
    te_items = list_split(spice, 'test')
    X_a, y_a, g_a = featurize(spice, 'trainval', tv_items, gw)
    X_b, y_b, g_b = featurize(spice, 'test', te_items, gw)
    X  = np.vstack([X_a, X_b])
    y7 = np.concatenate([y_a, y_b])
    g  = np.concatenate([g_a, g_b + len(tv_items)])   # unique photo ids across both parts
    photo_ids = np.unique(g)
    photo_y7  = np.array([y7[g == i][0] for i in photo_ids])
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    for task in ['7-class', '3-class']:
        lab = y7 if task == '7-class' else np.array([to_type(l) for l in y7])
        accs = []
        for tr_i, te_i in skf.split(photo_ids, photo_y7):
            tr_m, te_m = np.isin(g, photo_ids[tr_i]), np.isin(g, photo_ids[te_i])
            ids, cls, p = catboost_fit_predict(spice, X[tr_m], lab[tr_m], X[te_m], g[te_m])
            truth = np.array([lab[g == i][0] for i in ids])
            accs.append(accuracy_score(truth, cls[p.argmax(1)]) * 100)
        cv_rows.append({'spice': spice, 'task': task, 'folds': np.round(accs, 1).tolist(),
                        'mean': np.mean(accs), 'std': np.std(accs), 'min': np.min(accs), 'max': np.max(accs)})
        print(f'{spice:<22} {task}: {np.mean(accs):.1f} ± {np.std(accs):.1f}%  folds={np.round(accs, 1).tolist()}')

cv_df = pd.DataFrame(cv_rows).round(2)
cv_df.to_csv(os.path.join(OUT_DIR, 'catboost_5fold_cv.csv'), index=False)
display(cv_df)

## 8 — Verdict

In [ ]:
print('=' * 78)
print('  FAIR COMPARISON VERDICT (same held-out photos, photo-level accuracy)')
print('=' * 78)
for _, r in paired_df.iterrows():
    res = results[(results.spice == r.spice) & (results.task == r.task)].set_index('model')
    ra, ca = res.loc['ResNet50', 'acc'], res.loc['CatBoost pipeline', 'acc']
    sig = 'significant (p<0.05)' if r.mcnemar_p < 0.05 else 'NOT significant'
    winner = 'CatBoost' if ca > ra else ('ResNet50' if ra > ca else 'tie')
    print(f"{r.spice:<22} {r.task}: ResNet50 {ra:5.1f}% | CatBoost {ca:5.1f}% -> {winner:<8} "
          f"diff {r.CatBoost_minus_ResNet_pp:+.1f} pp [95% CI {r.diff_ci_low:+.1f}, {r.diff_ci_high:+.1f}], McNemar p={r.mcnemar_p:.3f} ({sig})")
print('\nNotes:')
print(' * Test sets are small (~56-57 photos per spice, ~8 per class), so CIs are wide.')
print(' * The CatBoost routing (Pipeline 2 for chilli) was chosen after seeing test results in the original')
print('   notebook; ResNet50 was trained once with no CV. Section 7 shows the CatBoost fold-to-fold spread.')
print(' * Only the 7-class task gives a concentration (5/10/15%) — needed for severity scoring.')

## 9 — Ensemble: let both models vote (7-class, keeps concentration)
Both models output a probability for each of the 7 classes. The ensemble blends them:
`p = w · ResNet50 + (1 − w) · CatBoost`, then picks the highest.

**No peeking at the test set.** The weight `w` is chosen per spice on the **validation photos only**
(CatBoost trained on train only, so its validation predictions are out-of-sample; ResNet50 never trained on
validation photos), by lowest log-loss over w ∈ {0, 0.1, …, 1}. Then CatBoost is retrained on train+val (as in
section 6) and everything is scored **once** on the same test photos. A fixed 50/50 blend is reported too, since
it involves no tuning at all. Self-contained: only needs the helper cells (sections 1–5).

In [ ]:
from sklearn.metrics import log_loss

WEIGHTS = np.round(np.arange(0, 1.01, 0.1), 2)   # weight on ResNet50; (1 - w) goes to CatBoost

def align(probs, classes, target):
    # Reorder probability columns into the `target` class order
    classes = list(classes)
    return probs[:, [classes.index(c) for c in target]]

ens_rows, chosen_w = [], {}
for spice in SPICES:
    gw = (spice == 'red_chilli_powder')
    train_items, val_items = list_split(spice, 'train'), list_split(spice, 'val')
    test_items = list_split(spice, 'test')
    X_tv, y_tv, g_tv = featurize(spice, 'trainval', train_items + val_items, gw)   # cached
    X_te, _,    g_te = featurize(spice, 'test', test_items, gw)                    # cached
    n_tr = len(train_items)
    tr_m = g_tv < n_tr

    # Step 1: choose w on validation photos only
    v_ids, c_cls_v, c_pv = catboost_fit_predict(spice, X_tv[tr_m], y_tv[tr_m], X_tv[~tr_m], g_tv[~tr_m] - n_tr)
    r_cls, r_pv = resnet_predict(spice, val_items)
    order = np.array(sorted(r_cls))
    r_pv = align(r_pv[v_ids], r_cls, order)
    c_pv = align(c_pv, c_cls_v, order)
    y_val = np.array([val_items[i][1] for i in v_ids])
    val_loss = [log_loss(y_val, w * r_pv + (1 - w) * c_pv, labels=order) for w in WEIGHTS]
    best_w = float(WEIGHTS[int(np.argmin(val_loss))])
    chosen_w[spice] = best_w
    print(f'{spice}: chosen w (ResNet50 share) = {best_w:.1f} | val log-loss by w: '
          + ', '.join(f'{w:.1f}:{l:.3f}' for w, l in zip(WEIGHTS, val_loss)))

    # Step 2: score once on the untouched test photos
    t_ids, c_cls_t, c_pt = catboost_fit_predict(spice, X_tv, y_tv, X_te, g_te)
    r_cls_t, r_pt = resnet_predict(spice, test_items)
    r_pt = align(r_pt[t_ids], r_cls_t, order)
    c_pt = align(c_pt, c_cls_t, order)
    y_te7 = np.array([test_items[i][1] for i in t_ids])
    y_te3 = np.array([to_type(l) for l in y_te7])

    candidates = {
        'ResNet50 alone':            r_pt,
        'CatBoost alone':            c_pt,
        'Ensemble 50/50':            0.5 * r_pt + 0.5 * c_pt,
        'Ensemble tuned on val':     best_w * r_pt + (1 - best_w) * c_pt,
    }
    correct_resnet = order[r_pt.argmax(1)] == y_te7
    for name, p in candidates.items():
        pred7 = order[p.argmax(1)]
        t3, p3 = collapse_to_type(order, p)
        pred3 = t3[p3.argmax(1)]
        correct = pred7 == y_te7
        lo, hi = boot_ci(correct)
        mp, only_res, only_this = mcnemar(correct_resnet, correct)
        ens_rows.append({'spice': spice, 'model': name, 'w_resnet': {'ResNet50 alone': 1.0, 'CatBoost alone': 0.0,
                         'Ensemble 50/50': 0.5}.get(name, best_w),
                         'acc_7class': correct.mean() * 100, 'ci_low': lo, 'ci_high': hi,
                         'acc_3class': accuracy_score(y_te3, pred3) * 100,
                         'macro_f1_7': f1_score(y_te7, pred7, average='macro', zero_division=0) * 100,
                         'pure_recall': recall_score(y_te7, pred7, labels=['pure'], average='macro', zero_division=0) * 100,
                         'vs_ResNet_only_ResNet_right': only_res, 'vs_ResNet_only_this_right': only_this,
                         'mcnemar_p_vs_ResNet': mp})

ens_df = pd.DataFrame(ens_rows).round(2)
ens_df.to_csv(os.path.join(OUT_DIR, 'ensemble_results.csv'), index=False)
display(ens_df)

print('\n' + '=' * 78)
print('  ENSEMBLE VERDICT (7-class accuracy on the same test photos; 3-class in brackets)')
print('=' * 78)
for spice in SPICES:
    sub = ens_df[ens_df.spice == spice].set_index('model')
    line = ' | '.join(f"{m}: {sub.loc[m, 'acc_7class']:.1f}% ({sub.loc[m, 'acc_3class']:.1f}%)" for m in sub.index)
    tuned = sub.loc['Ensemble tuned on val']
    gain = tuned.acc_7class - sub.loc['ResNet50 alone', 'acc_7class']
    sig = 'significant' if tuned.mcnemar_p_vs_ResNet < 0.05 else 'not significant'
    print(f'{spice} (w={chosen_w[spice]:.1f}): {line}')
    print(f'    tuned ensemble vs ResNet50 alone: {gain:+.1f} pp, McNemar p={tuned.mcnemar_p_vs_ResNet:.3f} ({sig})')